# Evaluation
## Testing datasets
### Retrieval ground truth

In [1]:
from wikifin_rag.ingest import load_wikifin_data

In [2]:
documents = load_wikifin_data()

In [3]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [4]:
data_gen_instructions = """
You emulate a student or young professional who has questions about personal finance.
Formulate {} questions this person might ask based on a document passage.
The passage should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [5]:
from dotenv import load_dotenv
from openai import OpenAI
from wikifin_rag.evaluation_utils import llm_structured_retry
import json

In [6]:
load_dotenv(override=True)
openai_client = OpenAI()

In [7]:
def generate_document_ground_truth(doc, n=5):
    user_prompt = doc['content']

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions.format(n),
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [8]:
from concurrent.futures import ThreadPoolExecutor
from wikifin_rag.evaluation_utils import map_progress, calculate_total_cost
import pandas as pd
from wikifin_rag.config import PROJECT_ROOT
from pathlib import Path
import numpy as np
import os

In [9]:
def generate_corpus_ground_truth(documents, n=5, file_path=PROJECT_ROOT / "data" / "evals" / "ground_truth-new.csv"):
    with ThreadPoolExecutor(max_workers=6) as pool:
        results = map_progress(pool, documents, lambda doc: generate_document_ground_truth(doc, n=n))

    ground_truth = []
    usages = []

    for records, usage in results:
        ground_truth.extend(records)
        usages.append(usage)

    # total_cost = calculate_total_cost(usages)

    file_path = Path(file_path)
    file_path.parent.mkdir(parents=True, exist_ok=True)
    df_ground_truth = pd.DataFrame(ground_truth)
    df_ground_truth.to_csv(file_path, index=False)


In [10]:
# number of documents to use
n_documents = 300

# number of question to generate per document
n_questions = 5

# file path for the generated ground truth dataset
ground_truth_file_path = PROJECT_ROOT / "data" / "evals" / "ground_truth-new.csv"

In [11]:
# TODO: change to True to force a ground truth data refresh
refresh_ground_truth = False

In [12]:
ground_truth_docs = np.random.choice(documents, size=n_documents, replace=False)

In [13]:
if refresh_ground_truth or not os.path.exists(ground_truth_file_path):
    generate_corpus_ground_truth(documents=ground_truth_docs, n=n_questions, file_path=ground_truth_file_path)

## Retrieval function evaluation
### Text Search

In [14]:
from wikifin_rag.evaluation_utils import evaluate
from wikifin_rag.ingest import build_text_index

In [15]:
ts_index = build_text_index(documents=documents)

In [16]:
def text_search(query, boost_dict=None, num_results=5):
    return ts_index.search(
        query,
        num_results=num_results,
        boost_dict=boost_dict
    )

In [17]:
df_ground_truth = pd.read_csv(ground_truth_file_path)
ground_truth = df_ground_truth.to_dict(orient="records")

In [18]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
import pickle

In [19]:
def ts_objective(params):
    metrics = evaluate(
        ground_truth=ground_truth,
        search_function=lambda query: text_search(query=query, boost_dict=params)
    )

    return {
        'loss': -round(metrics['mrr'], 4),
        'status': STATUS_OK,
        'metrics': metrics
    }

In [20]:
n_evals = 25

In [21]:
ts_search_space = {
    "title": hp.uniform("title", 0, 20),
    "section": hp.uniform("section", 0, 20),
    "content": hp.uniform("content", 0, 20)
}

In [22]:
ts_trials = Trials()

In [23]:
ts_best_params = fmin(
    fn=ts_objective,
    space=ts_search_space,
    algo=tpe.suggest,
    max_evals=n_evals,
    trials=ts_trials
)

  0%|          | 0/25 [00:00<?, ?trial/s, best loss=?]

  0%|          | 0/1500 [00:00<?, ?it/s]

  4%|▍         | 1/25 [00:08<03:13,  8.05s/trial, best loss: -0.4267]

  0%|          | 0/1500 [00:00<?, ?it/s]

  8%|▊         | 2/25 [00:16<03:05,  8.07s/trial, best loss: -0.4267]

...

 92%|█████████▏| 23/25 [03:46<00:20, 10.42s/trial, best loss: -0.4267]

  0%|          | 0/1500 [00:00<?, ?it/s]

 96%|█████████▌| 24/25 [03:56<00:10, 10.47s/trial, best loss: -0.4267]

  0%|          | 0/1500 [00:00<?, ?it/s]

100%|██████████| 25/25 [04:07<00:00,  9.89s/trial, best loss: -0.4267]


In [24]:
ts_params_file_path = PROJECT_ROOT / "data" / "evals" / "ts_params.pkl"
ts_params_file_path.parent.mkdir(parents=True, exist_ok=True)

with open(ts_params_file_path, "wb") as file:
    pickle.dump(ts_best_params, file)

In [25]:
ts_trials_data = [{
    'id': trial['tid'],
    'mrr': trial['result']['metrics']['mrr'],
    'hit_rate': trial['result']['metrics']['hit_rate'],
    'title_weight': trial['misc']['vals']['title'][0],
    'section_weight': trial['misc']['vals']['section'][0],
    'content_weight': trial['misc']['vals']['content'][0]
} for trial in ts_trials.trials]

In [26]:
ts_trials_df = pd.DataFrame(ts_trials_data)

In [27]:
ts_trials_df.head()

,id,mrr,hit_rate,title_weight,section_weight,content_weight
0,0,0.426656,0.553333,7.386541,7.756178,17.313598
1,1,0.426656,0.553333,12.958937,9.904273,5.973399
2,2,0.426656,0.553333,7.316620,9.988599,12.368944
3,3,0.426656,0.553333,19.211141,2.378245,12.685844
4,4,0.426656,0.553333,10.719013,4.380462,17.430857


### Vector Search

In [28]:
from wikifin_rag.ingest import build_vector_index
from wikifin_rag.db_utils import embedding_factory
from wikifin_rag.embedder import Embedder
from hyperopt.pyll import scope

In [29]:
documents = load_wikifin_data(embedding_factory)
embeddings = [doc.pop('embedding') for doc in documents]

In [30]:
embedder = Embedder()

In [31]:
def vector_search(index, query, num_results=5):
    query_vector = embedder.encode(query)
    return index.search(
        query_vector,
        num_results=num_results
    )

In [33]:
q = ground_truth[0]['question']

In [34]:
text_search(q)

[{'id': '9cb1560888d4b8d4_1250',
  'title': 'Wat te doen voor de erfenis als iemand overlijdt?',
  'section': 'Wat houdt de aangifte van de erfenis in?',
  'content': 'gifte van een erfenis kan complex zijn. Bovendien moet ze tijdig en correct gebeuren, zo niet dreig je hogere belastingen te moeten betalen. Het kan daarom nuttig zijn je te laten bijstaan door een notaris, zeker als de overledene een groot vermogen had en er verschillende erfgenamen zijn.\nJe vindt ',
  'source_url': 'https://www.wikifin.be/nl/erven/erven-en-successierechten/wat-te-doen-voor-de-erfenis-als-iemand-overlijdt'},
 {'id': '634dc177ec510e0f_750',
  'title': 'Wanneer je erfenis plannen?',
  'section': 'Wanneer je erfenis plannen?',
  'content': 'het kapitaal uit een\xa0groepsverzekering ontvangt, Hoe eerder, hoe beter, : sterven kan op elk moment.',
  'source_url': 'https://www.wikifin.be/nl/erven/erfenis-plannen/wanneer-je-erfenis-plannen'},
 {'id': '9cb1560888d4b8d4_1000',
  'title': 'Wat te doen voor de erf

In [31]:
def vs_objective(params):
    vs_index = build_vector_index(embeddings, documents, **params)

    metrics = evaluate(
        ground_truth=ground_truth,
        search_function=lambda query: vector_search(index=vs_index, query=query)
    )

    return {
        'loss': -round(metrics['mrr'], 4),
        'status': STATUS_OK,
        'metrics': metrics
    }

In [32]:
vs_search_space = hp.pchoice("ann", [
    # HNSW
    (0.8, {
        "mode": "hnsw",
        "m": hp.choice("hnsw_m", [8, 12, 16, 24, 32]),
        "ef_construction": hp.choice(
            "ef_construction",
            [50, 100, 200, 400],
        ),
        "ef_search": hp.choice(
            "ef_search",
            [10, 20, 40, 80, 160, 320],
        )
    }),

    # LSH
    (0.1, {
        "mode": "lsh",
        "n_tables": scope.int(hp.quniform("lsh_n_tables", 2, 32, 1)),
        "hash_size": scope.int(hp.quniform("lsh_hash_size", 8, 24, 1)),
        "n_probe": scope.int(hp.quniform("lsh_n_probe", 0, 2, 1)),
    }),

    # IVF
    (0.1, {
        "mode": "ivf",
        "n_clusters": hp.choice(
            "n_clusters",
            [None, 32, 64, 128, 256, 512]
        ),
        "n_probe_clusters": scope.int(hp.quniform(
            "n_probe_clusters", 1, 32, 1
        )),
    }),
])

In [ ]:
vs_trials = Trials()

In [34]:
vs_best_params = fmin(
    fn=vs_objective,
    space=vs_search_space,
    algo=tpe.suggest,
    max_evals=n_evals,
    trials=vs_trials
)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

  0%|          | 0/1500 [00:00<?, ?it/s]

  2%|▏         | 1/50 [00:15<12:30, 15.31s/trial, best loss: -0.478]

  0%|          | 0/1500 [00:00<?, ?it/s]

  4%|▍         | 2/50 [00:38<15:57, 19.96s/trial, best loss: -0.5733]

...

 96%|█████████▌| 48/50 [1:20:00<02:30, 75.10s/trial, best loss: -0.5768] 

  0%|          | 0/1500 [00:00<?, ?it/s]

 98%|█████████▊| 49/50 [1:20:22<00:59, 59.07s/trial, best loss: -0.5768]

  0%|          | 0/1500 [00:00<?, ?it/s]

100%|██████████| 50/50 [1:20:33<00:00, 96.67s/trial, best loss: -0.5768]


In [ ]:
vs_params_file_path = PROJECT_ROOT / "data" / "evals" / "vs_params.pkl"
vs_params_file_path.parent.mkdir(parents=True, exist_ok=True)

with open(vs_params_file_path, "wb") as file:
    pickle.dump(vs_best_params, file)

In [ ]:
vs_trials.trials[0]

### Hybrid search

In [ ]:
def compute_rrf(rank, k=60):
    return 1 / (k + rank)

In [ ]:
def elastic_search_hybrid_rrf(text_search, vector_search, query, num_results=5, k=60):
    ts_results = text_search(query, num_results)
    vs_results = vector_search(query, num_results)

    scores = {}
    doc_map = {}

    for i in range(num_results):
        ts_doc = ts_results[i]
        ts_key = ts_doc["id"]
        doc_map[ts_key] = ts_doc
        scores[ts_key] = scores.get(ts_key, 0) + compute_rrf(i, k)

        vs_doc = vs_results[i]
        vs_key = vs_doc["id"]
        doc_map[vs_key] = vs_doc
        scores[vs_key] = scores.get(vs_key, 0) + compute_rrf(i, k)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_map[key] for key, _ in ranked[:num_results]]

In [ ]:
openai_client.close()